# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 01.02 · Selección robusta y triangulación neuronal de longitud

Compara 15, 20, 25, 30 y 35 segundos mediante un perfil clásico decisorio y dos análisis neuronales confirmatorios.

La selección usa exclusivamente `validation`; consultar `test` para elegir longitud produciría sesgo de selección [1]. La métrica clásica y la de MiniLM es *average precision* macro de los cuatro daños, apropiada para clases desbalanceadas [2]. Los baselines clásicos reutilizan TF-IDF y estimadores de scikit-learn [3] [4]. MiniLM funciona como encoder congelado con una cabeza logística; su familia se fundamenta en destilación y representaciones multilingües [5] [6], y el checkpoint queda fijado por su tarjeta [7]. Ollama usa `gemma3:4b`, salida estructurada y el prompt operativo vigente en `config/prompt_operacional_ollama_v3_2.md` [8] [9]. El bootstrap remuestrea videos completos para preservar la dependencia entre ventanas pareadas [10] [11]; la comparación pareada y las hipótesis predeclaradas siguen recomendaciones de evaluación estadística en PLN [12]. El contraste complementario formula explícitamente la hipótesis direccional de no inferioridad [13]. La composición del panel enriquecido, las cuotas por daño, los márgenes de no inferioridad, la penalización de salidas inválidas y la jerarquía de decisión son elecciones locales. Las métricas heterogéneas nunca se promedian: el perfil clásico selecciona o conserva la longitud; MiniLM examina sensibilidad a representaciones neuronales y Ollama examina sensibilidad semántica y viabilidad operativa. Si una familia diverge, se reporta el conflicto y se mantiene la decisión clásica hasta una validación humana independiente. `test` permanece cerrado.

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

In [ ]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('No se encontró pyproject.toml')

ROOT = find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from moderacion_peru.notebook_ui import show_callout, show_command, show_result, show_summary, show_table
OPERATIONAL_PROMPT=ROOT/'config/prompt_operacional_ollama_v3_2.md'
if not OPERATIONAL_PROMPT.is_file():
    raise FileNotFoundError(f'Falta el prompt operacional vigente: {OPERATIONAL_PROMPT}')
show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local', 'prompt_operacional': OPERATIONAL_PROMPT}, tone='success')


## Controles y protocolo predeclarado

In [ ]:
RUN_CHUNK_LENGTH_SMOKE_TEST=False
RUN_CHUNK_LENGTH_CONFIRMATORY_TEST=False
RUN_CHUNK_LENGTH_ROBUST_TEST=False  # Active solo para reconstruir la etapa clásica
RUN_NEURAL_ROBUST_TEST=True
RUN_MINILM_20_30_NONINFERIORITY_TEST=True
FORCE_NEURAL_ROBUST_RECOMPUTE=False
FORCE_MINILM_20_30_RECOMPUTE=False
CANDIDATE_SECONDS=(15,20,25,30,35)
TOY_MODELS=('complement_nb','sgd_incremental')
TOY_VIDEO_LIMITS={'train':40,'validation':16,'test':16}
TOY_MAX_FEATURES=12000
CONFIRMATORY_MODELS=('complement_nb','logistic_regression','sgd_incremental')
CONFIRMATORY_VIDEO_LIMITS={'train':200,'validation':80,'test':80}
CONFIRMATORY_SEEDS=(20260805,20260817,20260829)
CONFIRMATORY_MAX_FEATURES=20000
ROBUST_VIDEO_LIMITS={'train':300,'validation':100,'test':100}
ROBUST_SEEDS=(20260805,20260817,20260829,20260841,20260853)
ROBUST_MAX_FEATURES=25000
ROBUST_REFERENCE_SECONDS=30.0
ROBUST_NONINFERIORITY_MARGIN=0.01
ROBUST_BOOTSTRAP_REPLICATES=1000
ROBUST_CONFIDENCE_LEVEL=0.95
ROBUST_BOOTSTRAP_SEED=20260807
ROBUST_RUNTIME_BUDGET_SECONDS=1800.0
MAX_VALIDATION_AP_DROP=0.02
NEURAL_PANEL_SIZE=100
NEURAL_MIN_DAMAGE_PER_LABEL=20
NEURAL_MAX_ANCHORS_PER_VIDEO=2
NEURAL_REPORTING_COHORTS=5
NEURAL_PANEL_SELECTION_SEED=20260807
NEURAL_MINILM_MODEL='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
NEURAL_MINILM_REVISION='e8f8c211226b894fcb81acc59f3b34ba3efd5f42'
NEURAL_MINILM_TRAIN_LIMIT=1000
NEURAL_MINILM_BATCH_SIZE=16
NEURAL_MINILM_MAX_LENGTH=128
NEURAL_MINILM_BOOTSTRAP_REPLICATES=2000
NEURAL_MINILM_NONINFERIORITY_MARGIN=0.01
NEURAL_OLLAMA_MODEL='gemma3:4b'
NEURAL_OLLAMA_TIMEOUT_SECONDS=90.0
NEURAL_OLLAMA_MAX_WALL_SECONDS=5400.0
NEURAL_OLLAMA_RETRIES=1
NEURAL_OLLAMA_BOOTSTRAP_REPLICATES=2000
NEURAL_OLLAMA_NONINFERIORITY_MARGIN=0.02
NEURAL_OLLAMA_MINIMUM_SCHEMA_RATE=0.95
MINILM_NI_PANEL_SIZE=750
MINILM_NI_MIN_DAMAGE_VIDEOS=80
MINILM_NI_FOLDS=5
MINILM_NI_PANEL_SELECTION_SEED=20260831
MINILM_NI_REPEAT_SEEDS=(20260901,20260913,20260925)
MINILM_NI_TRAIN_LIMIT=4000
MINILM_NI_BOOTSTRAP_REPLICATES=5000
MINILM_NI_BOOTSTRAP_SEED=20260907
MANUAL_CHUNK_SECONDS=30.0
USE_ROBUST_RECOMMENDATION=True
APPLY_CHUNK_SELECTION=False
from moderacion_peru.colab import prepare_local_bundle_input
from moderacion_peru.chunk_optimization import activate_chunking_configuration, run_chunk_length_confirmatory_test, run_chunk_length_robust_test, run_chunk_length_smoke_test
from moderacion_peru.neural_chunk_robust import run_minilm_20_30_noninferiority_test, run_neural_chunk_robust_test
from moderacion_peru.incremental import DEFAULT_CHUNKING_CONFIGURATION
import json
def sig2(value):
    return None if value is None else float(f'{float(value):.2g}')
TRANSCRIPTS=ROOT/'datos/raw/transcripts_raw.jsonl'
CHUNKS_CHECKPOINT=prepare_local_bundle_input('chunks_v2',project_root=ROOT)
CHUNKS=Path(CHUNKS_CHECKPOINT['path'])
DATASET_CHECKPOINT=prepare_local_bundle_input('dataset_5_salidas',project_root=ROOT)
DATASET=Path(DATASET_CHECKPOINT['path'])
PILOT_ROOT=ROOT/'resultados/pilotos/chunk_length'
ROBUST_ROOT=PILOT_ROOT/'robust_30min'
NEURAL_ROOT=PILOT_ROOT/'neural_robust'
MINILM_NI_ROOT=NEURAL_ROOT/'minilm_20_30_noninferiority'
ROBUST_RESULT_PATH=ROBUST_ROOT/'robust_comparison.json'
NEURAL_RESULT_PATH=NEURAL_ROOT/'neural_robust_comparison.json'
MINILM_NI_RESULT_PATH=MINILM_NI_ROOT/'minilm_20_30_noninferiority.json'
ROBUST_RECOMMENDATION=ROBUST_ROOT/'robust_recommendation.json'
if tuple(CANDIDATE_SECONDS)!=(15,20,25,30,35):
    raise ValueError('Este protocolo exige CANDIDATE_SECONDS=(15,20,25,30,35)')
if RUN_CHUNK_LENGTH_CONFIRMATORY_TEST and RUN_CHUNK_LENGTH_ROBUST_TEST:
    raise ValueError('El perfil robusto ya incluye la confirmación; active solo uno')
show_summary('Secuencia configurada',{'1_clasico_decisorio':RUN_CHUNK_LENGTH_ROBUST_TEST or ROBUST_RESULT_PATH.is_file(),'2_minilm_confirmatorio':RUN_NEURAL_ROBUST_TEST,'3_ollama_confirmatorio':RUN_NEURAL_ROBUST_TEST,'4_minilm_no_inferioridad_20_30':RUN_MINILM_20_30_NONINFERIORITY_TEST,'longitudes_perfil':CANDIDATE_SECONDS,'contraste_complementario':(20,30),'panel_validation':NEURAL_PANEL_SIZE,'panel_crossfit_train':MINILM_NI_PANEL_SIZE,'cohortes_reporte':NEURAL_REPORTING_COHORTS,'respuestas_ollama_previstas':NEURAL_PANEL_SIZE*len(CANDIDATE_SECONDS),'test_usado_para_seleccion':False},tone='neutral')

## Diagnósticos clásicos opcionales

In [ ]:
if RUN_CHUNK_LENGTH_SMOKE_TEST:
    smoke_result=run_chunk_length_smoke_test(TRANSCRIPTS,CHUNKS,DATASET,PILOT_ROOT,candidate_seconds=CANDIDATE_SECONDS,model_names=TOY_MODELS,video_limits=TOY_VIDEO_LIMITS,max_features=TOY_MAX_FEATURES,max_validation_ap_drop=MAX_VALIDATION_AP_DROP)
    show_result('Recomendación exploratoria',smoke_result['recommendation'],tone='success')
    show_table('Smoke clásico',smoke_result['comparisons'],max_rows=len(CANDIDATE_SECONDS))
else:
    show_callout('Smoke clásico omitido','Es opcional y no sustituye el perfil robusto.',tone='neutral')
if RUN_CHUNK_LENGTH_CONFIRMATORY_TEST:
    confirmatory_result=run_chunk_length_confirmatory_test(TRANSCRIPTS,CHUNKS,DATASET,PILOT_ROOT,candidate_seconds=CANDIDATE_SECONDS,model_names=CONFIRMATORY_MODELS,video_limits=CONFIRMATORY_VIDEO_LIMITS,seeds=CONFIRMATORY_SEEDS,max_features=CONFIRMATORY_MAX_FEATURES)
    show_result('Recomendación confirmatoria corta',confirmatory_result['recommendation'],tone='success')
    show_table('Confirmación clásica corta',confirmatory_result['aggregated_comparisons'],max_rows=len(CANDIDATE_SECONDS))
else:
    show_callout('Confirmación corta omitida','Es opcional porque el perfil robusto ya incorpora cinco cohortes.',tone='neutral')

## Etapa 1 — perfil robusto clásico decisorio

In [ ]:
if RUN_CHUNK_LENGTH_ROBUST_TEST:
    robust_result=run_chunk_length_robust_test(TRANSCRIPTS,CHUNKS,DATASET,ROBUST_ROOT,candidate_seconds=CANDIDATE_SECONDS,reference_seconds=ROBUST_REFERENCE_SECONDS,model_names=CONFIRMATORY_MODELS,video_limits=ROBUST_VIDEO_LIMITS,seeds=ROBUST_SEEDS,max_features=ROBUST_MAX_FEATURES,bootstrap_replicates=ROBUST_BOOTSTRAP_REPLICATES,confidence_level=ROBUST_CONFIDENCE_LEVEL,noninferiority_margin=ROBUST_NONINFERIORITY_MARGIN,bootstrap_seed=ROBUST_BOOTSTRAP_SEED,runtime_budget_seconds=ROBUST_RUNTIME_BUDGET_SECONDS)
elif ROBUST_RESULT_PATH.is_file():
    robust_result=json.loads(ROBUST_RESULT_PATH.read_text(encoding='utf-8-sig'))
else:
    robust_result=None
if robust_result:
    classical_rows=[{'longitud_s':row['chunk_seconds'],'AP_validation':sig2(row['paired_validation_ap_macro_damage']),'IC95_AP':[sig2(row['bootstrap_ap_ci_low']),sig2(row['bootstrap_ap_ci_high'])],'delta_vs_30s':sig2(row['delta_vs_reference']),'IC95_delta':[sig2(row['delta_vs_reference_ci_low']),sig2(row['delta_vs_reference_ci_high'])],'no_inferior':'Sí' if row['noninferior'] else 'No','proxy_costo':row['compute_proxy']} for row in robust_result['bootstrap']['comparisons']]
    show_table('Perfil clásico: resultados reportables',classical_rows,max_rows=len(CANDIDATE_SECONDS))
    show_summary('Decisión primaria',{'longitud_s':robust_result['recommendation']['recommended_seconds'],'partición':'validation','métrica':'AP macro de cuatro daños','cohortes':robust_result['design']['paired_cohorts'],'ajustes':robust_result['design']['fits'],'réplicas_bootstrap':robust_result['bootstrap']['replicates'],'test_usado_para_seleccion':False,'artefacto':ROBUST_RESULT_PATH},tone='success')
else:
    show_callout('Falta el perfil clásico','Active RUN_CHUNK_LENGTH_ROBUST_TEST=True antes del perfil neuronal.',tone='danger')

## Etapas 2 y 3 — perfil neuronal robusto pareado

In [ ]:
if NEURAL_RESULT_PATH.is_file() and not FORCE_NEURAL_ROBUST_RECOMPUTE:
    neural_result=json.loads(NEURAL_RESULT_PATH.read_text(encoding='utf-8-sig'))
    show_callout('Perfil neuronal cargado','Se leyó el JSON consolidado; no se llamó MiniLM ni Ollama.',tone='success')
elif RUN_NEURAL_ROBUST_TEST:
    if not ROBUST_RESULT_PATH.is_file():
        raise FileNotFoundError('Ejecute primero la etapa clásica robusta')
    show_callout('Perfil neuronal en ejecución','La celda escribe checkpoints durante MiniLM y Ollama. Las tablas aparecen al final; una nueva ejecución reutiliza todo resultado con firma compatible.',tone='neutral')
    neural_result=run_neural_chunk_robust_test(TRANSCRIPTS,CHUNKS,DATASET,ROBUST_ROOT,NEURAL_ROOT,candidate_seconds=CANDIDATE_SECONDS,reference_seconds=ROBUST_REFERENCE_SECONDS,seeds=ROBUST_SEEDS,panel_size=NEURAL_PANEL_SIZE,minimum_damage_anchors_per_label=NEURAL_MIN_DAMAGE_PER_LABEL,max_anchors_per_video=NEURAL_MAX_ANCHORS_PER_VIDEO,reporting_cohorts=NEURAL_REPORTING_COHORTS,panel_selection_seed=NEURAL_PANEL_SELECTION_SEED,minilm_model_id=NEURAL_MINILM_MODEL,minilm_revision=NEURAL_MINILM_REVISION,minilm_train_limit_per_cohort=NEURAL_MINILM_TRAIN_LIMIT,minilm_batch_size=NEURAL_MINILM_BATCH_SIZE,minilm_max_length=NEURAL_MINILM_MAX_LENGTH,minilm_bootstrap_replicates=NEURAL_MINILM_BOOTSTRAP_REPLICATES,minilm_noninferiority_margin=NEURAL_MINILM_NONINFERIORITY_MARGIN,ollama_model=NEURAL_OLLAMA_MODEL,ollama_timeout_seconds=NEURAL_OLLAMA_TIMEOUT_SECONDS,ollama_max_wall_seconds=NEURAL_OLLAMA_MAX_WALL_SECONDS,ollama_retries=NEURAL_OLLAMA_RETRIES,ollama_bootstrap_replicates=NEURAL_OLLAMA_BOOTSTRAP_REPLICATES,ollama_noninferiority_margin=NEURAL_OLLAMA_NONINFERIORITY_MARGIN,ollama_minimum_schema_rate=NEURAL_OLLAMA_MINIMUM_SCHEMA_RATE,confidence_level=ROBUST_CONFIDENCE_LEVEL)
else:
    neural_result=None
if neural_result:
    panel=neural_result['panel']
    show_summary('Panel pareado de validation',{'anclas':panel['anchors'],'videos':panel['distinct_videos'],'conteos_etiqueta':panel['label_counts'],'cohortes_disjuntas':panel['anchors_per_reporting_cohort'],'muestra':'enriquecida; no estima prevalencia','test_usado':False},tone='neutral')
    minilm_rows=[{'longitud_s':row['chunk_seconds'],'AP_ensemble':sig2(row['ensemble_validation_ap_macro_damage']),'IC95_AP':[sig2(row['bootstrap_ap_ci_low']),sig2(row['bootstrap_ap_ci_high'])],'delta_vs_30s':sig2(row['delta_vs_reference']),'IC95_delta':[sig2(row['delta_vs_reference_ci_low']),sig2(row['delta_vs_reference_ci_high'])],'no_inferior':'Sí' if row['noninferior'] else 'No'} for row in neural_result['minilm']['bootstrap']['comparisons']]
    show_table('MiniLM robusto: resultados reportables',minilm_rows,max_rows=len(CANDIDATE_SECONDS))
    ollama_boot={float(row['chunk_seconds']):row for row in neural_result['ollama']['bootstrap']['comparisons']}
    ollama_rows=[]
    for row in neural_result['ollama']['duration_results']:
        boot=ollama_boot[float(row['chunk_seconds'])]
        ollama_rows.append({'longitud_s':row['chunk_seconds'],'válidas':f"{row['successful_rows']}/{row['requested_rows']}",'tasa_esquema':sig2(row['valid_schema_rate']),'F1_macro_daños':sig2(row['f1_macro_damage']),'IC95_F1':[sig2(boot['bootstrap_f1_ci_low']),sig2(boot['bootstrap_f1_ci_high'])],'delta_vs_30s':sig2(boot['delta_vs_reference']),'IC95_delta':[sig2(boot['delta_vs_reference_ci_low']),sig2(boot['delta_vs_reference_ci_high'])],'exact_match':sig2(row['exact_label_set_match_rate']),'hamming_loss':sig2(row['hamming_loss_five'])})
    show_table('Ollama robusto: resultados reportables',ollama_rows,max_rows=len(CANDIDATE_SECONDS))
    hierarchy=neural_result['hierarchical_synthesis']
    hierarchy_warning='conflict' in hierarchy['hierarchy_status'] or neural_result['reporting_status']!='complete'
    show_summary('Síntesis jerárquica',hierarchy,tone='warning' if hierarchy_warning else 'success')
    if neural_result['reporting_status']!='complete':
        show_callout('Ejecución parcial y reanudable','Vuelva a ejecutar esta celda. Se conservarán respuestas válidas y ajustes MiniLM cuya firma coincida.',tone='warning')
else:
    show_callout('Perfil neuronal pendiente','Active RUN_NEURAL_ROBUST_TEST=True después del perfil clásico. El diseño solicita 25 cabezas MiniLM y 500 respuestas Ollama.',tone='neutral')

## Etapa 4 — no inferioridad MiniLM 20 s frente a 30 s

In [ ]:
# Esta etapa resuelve el resultado MiniLM inconcluso del panel piloto.
# No repite las cinco longitudes: usa predicción fuera de pliegue por video
# sobre train, mantiene test cerrado y conserva 30 s como decisión clásica.
if MINILM_NI_RESULT_PATH.is_file() and not FORCE_MINILM_20_30_RECOMPUTE:
    minilm_ni_result=json.loads(MINILM_NI_RESULT_PATH.read_text(encoding='utf-8-sig'))
    show_callout('Contraste 20 s–30 s cargado','Se leyó el JSON consolidado; no se recalcularon embeddings, ajustes ni bootstrap.',tone='success')
elif RUN_MINILM_20_30_NONINFERIORITY_TEST:
    show_callout('Contraste 20 s–30 s en ejecución','Primera corrida estimada: 12–20 min en CPU. El resultado final y cada pliegue son reanudables por firma.',tone='neutral')
    minilm_ni_result=run_minilm_20_30_noninferiority_test(TRANSCRIPTS,CHUNKS,DATASET,ROBUST_ROOT,MINILM_NI_ROOT,panel_size=MINILM_NI_PANEL_SIZE,minimum_damage_videos_per_label=MINILM_NI_MIN_DAMAGE_VIDEOS,folds=MINILM_NI_FOLDS,panel_selection_seed=MINILM_NI_PANEL_SELECTION_SEED,classical_seeds=ROBUST_SEEDS,repeat_seeds=MINILM_NI_REPEAT_SEEDS,model_id=NEURAL_MINILM_MODEL,revision=NEURAL_MINILM_REVISION,train_limit_per_fit=MINILM_NI_TRAIN_LIMIT,batch_size=NEURAL_MINILM_BATCH_SIZE,max_length=NEURAL_MINILM_MAX_LENGTH,bootstrap_replicates=MINILM_NI_BOOTSTRAP_REPLICATES,confidence_level=ROBUST_CONFIDENCE_LEVEL,noninferiority_margin=NEURAL_MINILM_NONINFERIORITY_MARGIN,bootstrap_seed=MINILM_NI_BOOTSTRAP_SEED)
else:
    minilm_ni_result=None
if minilm_ni_result:
    ni_rows=[{'longitud_s':row['chunk_seconds'],'AP_OOF':sig2(row['ensemble_validation_ap_macro_damage']),'IC95_AP':[sig2(row['bootstrap_ap_ci_low']),sig2(row['bootstrap_ap_ci_high'])],'delta_vs_30s':sig2(row['delta_vs_reference']),'IC95_delta':[sig2(row['delta_vs_reference_ci_low']),sig2(row['delta_vs_reference_ci_high'])],'no_inferior':'Sí' if row['noninferior'] else 'No'} for row in minilm_ni_result['bootstrap']['comparisons']]
    show_table('MiniLM 20 s–30 s: contraste reportable',ni_rows,max_rows=2)
    ni=minilm_ni_result['interpretation']
    show_summary('Conclusión de no inferioridad',{'estado':ni['status'],'hipótesis_nula':ni['null_hypothesis'],'margen':ni['noninferiority_margin'],'delta_AP_20_menos_30':sig2(ni['delta_ap']),'IC95_delta':[sig2(ni['delta_ap_ci_low']),sig2(ni['delta_ap_ci_high'])],'no_inferior':'Sí' if ni['noninferior'] else 'No','superioridad_demostrada':'Sí' if ni['superiority_established'] else 'No','videos':minilm_ni_result['design']['panel_video_clusters'],'pliegues':minilm_ni_result['design']['folds'],'repeticiones':minilm_ni_result['design']['training_repeats'],'test_usado':False,'efecto_decisorio':'Ninguno; 30 s continúa como selección clásica'},tone='success' if ni['noninferior'] else 'warning')
else:
    show_callout('Contraste complementario omitido','El panel neuronal original permanece inconcluso; active RUN_MINILM_20_30_NONINFERIORITY_TEST=True.',tone='warning')

## Lectura académica y límites de aplicación

In [ ]:
if neural_result:
    show_summary('Roles no intercambiables',{'clásico':'decisorio; selecciona o conserva longitud','MiniLM robusto':'confirmatorio; sensibilidad a representación neuronal continua','MiniLM 20/30 OOF':'complementario; contrasta no inferioridad interna sin cambiar la selección','Ollama':'confirmatorio; sensibilidad semántica y factibilidad de salida estructurada','agregación_entre_familias':'ninguna','política_de_conflicto':'conservar la selección clásica hasta validación humana independiente','artefacto_perfil':NEURAL_RESULT_PATH,'artefacto_no_inferioridad':MINILM_NI_RESULT_PATH},tone='neutral')
    ni_followup=globals().get('minilm_ni_result')
    if ni_followup:
        ni=ni_followup['interpretation']
        show_summary('Conclusión conjunta actualizada',{'otra_longitud_demostró_ser_mejor_que_30s':False,'20s_no_inferior_a_30s_en_MiniLM':'Sí' if ni['noninferior'] else 'No','20s_superior_a_30s_en_MiniLM':'Sí' if ni['superiority_established'] else 'No','selección_principal_s':30,'justificación':'La evidencia complementaria no reemplaza el perfil clásico decisorio.'},tone='success')
    show_callout('Alcance','Los intervalos describen este panel enriquecido de validation. No estiman prevalencia ni desempeño productivo. La confianza declarada por Ollama no se interpreta como probabilidad calibrada.',tone='warning')
else:
    show_callout('Resultados aún no ejecutados','No reporte expectativas como resultados. Ejecute la etapa neuronal y use su artefacto canónico.',tone='neutral')

## Activación manual y reversible

In [ ]:
if USE_ROBUST_RECOMMENDATION:
    if not ROBUST_RECOMMENDATION.is_file():
        raise FileNotFoundError('Ejecute primero el perfil robusto clásico')
    selected_seconds=float(json.loads(ROBUST_RECOMMENDATION.read_text(encoding='utf-8-sig'))['recommended_seconds'])
    selection_source='01_02_robust_bootstrap_recommendation'
else:
    selected_seconds=float(MANUAL_CHUNK_SECONDS)
    selection_source='01_02_manual'
selected_config={**DEFAULT_CHUNKING_CONFIGURATION,'max_seconds':selected_seconds}
if APPLY_CHUNK_SELECTION:
    activation=activate_chunking_configuration(ROOT,selected_config,source=selection_source)
    show_result('Configuración activada sin borrar derivados',activation,tone='success')
else:
    show_summary('Selección previsualizada',{'segundos':selected_seconds,'origen':selection_source,'regla':'Las pruebas neuronales no modifican automáticamente este valor.','acción':'Active APPLY_CHUNK_SELECTION=True; 01_03 materializará o restaurará la firma.'},tone='neutral')

## Referencias

[1] G. C. Cawley and N. L. C. Talbot, "On Over-Fitting in Model Selection and Subsequent Selection Bias in Performance Evaluation," J. Mach. Learn. Res., vol. 11, pp. 2079–2107, 2010.

[2] T. Saito and M. Rehmsmeier, "The Precision-Recall Plot Is More Informative than the ROC Plot When Evaluating Binary Classifiers on Imbalanced Datasets," PLOS ONE, vol. 10, no. 3, Art. no. e0118432, 2015, doi: 10.1371/journal.pone.0118432.

[3] G. Salton and C. Buckley, "Term-Weighting Approaches in Automatic Text Retrieval," Inf. Process. Manage., vol. 24, no. 5, pp. 513–523, 1988, doi: 10.1016/0306-4573(88)90021-0.

[4] F. Pedregosa, G. Varoquaux, A. Gramfort, et al., "Scikit-Learn: Machine Learning in Python," J. Mach. Learn. Res., vol. 12, pp. 2825–2830, 2011. [Online]. Available: https://www.jmlr.org/papers/v12/pedregosa11a.html

[5] W. Wang, F. Wei, L. Dong, et al., "MiniLM: Deep Self-Attention Distillation for Task-Agnostic Compression of Pre-Trained Transformers," in Adv. Neural Inf. Process. Syst., vol. 33, 2020. [Online]. Available: https://proceedings.neurips.cc/paper/2020/hash/3f5ee243547dee91fbd053c1c4a845aa-Abstract.html

[6] N. Reimers and I. Gurevych, "Making Monolingual Sentence Embeddings Multilingual Using Knowledge Distillation," in Proc. EMNLP, 2020, pp. 4512–4525, doi: 10.18653/v1/2020.emnlp-main.365.

[7] Sentence Transformers, "Model Card: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2," Hugging Face Hub, revision e8f8c211226b894fcb81acc59f3b34ba3efd5f42, 2026. [Online]. Available: https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/tree/e8f8c211226b894fcb81acc59f3b34ba3efd5f42

[8] Ollama, "Model Card: gemma3:4b," Ollama Model Library, 2026. [Online]. Available: https://ollama.com/library/gemma3:4b. Accessed: Aug. 6, 2026.

[9] Ollama, "Structured Outputs," Ollama Documentation, 2026. [Online]. Available: https://docs.ollama.com/capabilities/structured-outputs. Accessed: Aug. 5, 2026.

[10] B. Efron, "Bootstrap Methods: Another Look at the Jackknife," The Annals of Statistics, vol. 7, no. 1, pp. 1–26, 1979, doi: 10.1214/aos/1176344552.

[11] C. A. Field and A. H. Welsh, "Bootstrapping Clustered Data," Journal of the Royal Statistical Society: Series B, vol. 69, no. 3, pp. 369–390, 2007, doi: 10.1111/j.1467-9868.2007.00593.x.

[12] R. Dror, G. Baumer, S. Shlomov, et al., "The Hitchhiker's Guide to Testing Statistical Significance in Natural Language Processing," in Proc. 56th Annual Meeting ACL, 2018, pp. 1383–1392, doi: 10.18653/v1/P18-1128.

[13] W. C. Blackwelder, "Proving the Null Hypothesis in Clinical Trials," Controlled Clinical Trials, vol. 3, no. 4, pp. 345–353, 1982, doi: 10.1016/0197-2456(82)90024-1.